# LeetCode 700 — Search in a Binary Search Tree bằng Python

**Link đề:** https://leetcode.com/problems/search-in-a-binary-search-tree/
**Độ khó:** Easy · **Chủ đề:** Tree, Binary Search Tree, Binary Tree

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def searchBST(self, root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
        ...
```

## Mục tiêu học
1. Hiểu cây nhị phân (binary tree), node, con trái/phải, `None`, và cách LeetCode biểu diễn cây bằng list `[4,2,7,1,3]`.
2. Hiểu **tính chất BST**: mọi giá trị ở cây con trái nhỏ hơn node, cây con phải lớn hơn node.
3. Đi từ cách duyệt **toàn bộ cây** (không dùng tính chất BST) tới **tìm theo một đường đi** (dùng BST) — đệ quy và vòng lặp.
4. Trace đường đi tìm kiếm, chứng minh đúng, phân tích `O(h)` với `h` là chiều cao cây.
5. Áp dụng sang biến thể: chèn vào BST, tìm min/max, tìm floor/ceiling.

## 1. Đề bài và ví dụ

Cho `root` của một **binary search tree (BST)** và số nguyên `val`. Tìm node có giá trị bằng `val` và trả về **cây con gốc tại node đó**. Nếu không có, trả về `null` (Python: `None`).

| `root` | `val` | Output | Ý nghĩa |
|---|:---:|---|---|
| `[4,2,7,1,3]` | `2` | `[2,1,3]` | node 2 cùng hai con 1 và 3 |
| `[4,2,7,1,3]` | `5` | `[]` | không có node 5 |

```text
        4
       / \
      2   7
     / \
    1   3
```

**Constraints:** số node trong `[1, 5000]`; `1 <= Node.val <= 10^7`; `root` là BST; `1 <= val <= 10^7`.

Chú ý: ta trả về **node** (đối tượng), không phải giá trị số. LeetCode tự in node đó thành list `[2,1,3]`.

## 2. Nền tảng: class `TreeNode` và đối tượng trong Python

### 2.1. Class là "khuôn", object là "bánh"

`class TreeNode` mô tả một node có 3 thuộc tính: `val` (giá trị), `left` (con trái), `right` (con phải). `None` nghĩa là "không có con".

`__init__` là hàm khởi tạo, chạy khi ta viết `TreeNode(4)`. `self` là chính đối tượng đang được tạo.

In [1]:
from typing import Optional, List

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
    def __repr__(self):
        return f"TreeNode({self.val})"

leaf = TreeNode(1)
print(leaf, leaf.val, leaf.left, leaf.right)

TreeNode(1) 1 None None


`__repr__` cho Python biết cách in đối tượng; nhờ nó `print(node)` hiện `TreeNode(1)` thay vì địa chỉ bộ nhớ khó đọc.

### 2.2. Tạo cây bằng tay

In [2]:
root = TreeNode(4,
                TreeNode(2, TreeNode(1), TreeNode(3)),
                TreeNode(7))
print("gốc:", root.val)
print("con trái của gốc:", root.left.val, "| con phải của gốc:", root.right.val)
print("con trái của 2:", root.left.left.val, "| con phải của 2:", root.left.right.val)
print("con của 7:", root.right.left, root.right.right)

gốc: 4
con trái của gốc: 2 | con phải của gốc: 7
con trái của 2: 1 | con phải của 2: 3
con của 7: None None


### 2.3. Biểu diễn level-order của LeetCode và hàm dựng/in cây

LeetCode viết cây theo **từng tầng, từ trái sang phải**, `null` cho vị trí trống: `[4,2,7,1,3]`. Ta viết 2 hàm:
- `build_tree(values)` dựng cây từ list (dùng **hàng đợi** `deque`).
- `tree_to_list(root)` làm ngược lại để so sánh kết quả.

`collections.deque` là hàng đợi: `append` thêm vào cuối, `popleft` lấy ra từ đầu — đều `O(1)`.

In [3]:
from collections import deque

q = deque([1, 2])
q.append(3)
print(q.popleft(), q)   # 1 deque([2, 3])

1 deque([2, 3])


In [4]:
def build_tree(values: list) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    it = iter(values)
    root = TreeNode(next(it))
    queue = deque([root])
    while queue:
        node = queue.popleft()
        try:
            v = next(it)
            if v is not None:
                node.left = TreeNode(v)
                queue.append(node.left)
            v = next(it)
            if v is not None:
                node.right = TreeNode(v)
                queue.append(node.right)
        except StopIteration:
            break
    return root


def tree_to_list(root: Optional[TreeNode]) -> list:
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:     # bỏ các None thừa ở cuối
        out.pop()
    return out


root = build_tree([4, 2, 7, 1, 3])
print(tree_to_list(root))
print(tree_to_list(root.left))
print(tree_to_list(None))

[4, 2, 7, 1, 3]
[2, 1, 3]
[]


Giải thích nhanh:
- `iter(values)` tạo "con trỏ" đọc lần lượt; `next(it)` lấy phần tử kế tiếp, hết thì ném `StopIteration`.
- `try ... except StopIteration: break` bắt lỗi "hết phần tử" để dừng.

Thêm hàm in cây dạng nghiêng (con phải ở trên) cho dễ nhìn:

In [5]:
def show(root: Optional[TreeNode], indent: int = 0) -> None:
    if root is None:
        return
    show(root.right, indent + 4)
    print(" " * indent + str(root.val))
    show(root.left, indent + 4)

show(build_tree([4, 2, 7, 1, 3]))

    7
4
        3
    2
        1


## 3. Nền tảng: tính chất Binary Search Tree

BST là cây nhị phân mà **với mọi node**:
- mọi giá trị trong **cây con trái** `<` giá trị node;
- mọi giá trị trong **cây con phải** `>` giá trị node.

Hệ quả quan trọng: duyệt **inorder** (trái → node → phải) cho dãy **tăng dần**.

In [6]:
def inorder(root: Optional[TreeNode]) -> list:
    if root is None:
        return []
    return inorder(root.left) + [root.val] + inorder(root.right)

print(inorder(build_tree([4, 2, 7, 1, 3])))
print(inorder(build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13])))

[1, 2, 3, 4, 7]
[1, 3, 4, 6, 7, 8, 10, 13, 14]


### 3.1. Đệ quy (recursion) là gì?

Hàm **gọi chính nó** trên bài toán nhỏ hơn. Cần 2 phần:
1. **Base case** (trường hợp dừng): ví dụ `root is None`.
2. **Bước đệ quy**: giải bằng kết quả của cây con.

Ví dụ đếm số node:

In [7]:
def count_nodes(root):
    if root is None:            # base case
        return 0
    return 1 + count_nodes(root.left) + count_nodes(root.right)

print(count_nodes(build_tree([4, 2, 7, 1, 3])))   # 5

5


## 4. Cách 1 — Duyệt toàn bộ cây (không dùng tính chất BST)

Người mới thường nghĩ: "đi qua mọi node, gặp node có `val` thì trả về". Cách này đúng với **mọi cây nhị phân**, nhưng phải thăm tới `n` node.

In [8]:
def search_any_tree(root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
    if root is None:
        return None
    if root.val == val:
        return root
    found = search_any_tree(root.left, val)
    if found is not None:
        return found
    return search_any_tree(root.right, val)

root = build_tree([4, 2, 7, 1, 3])
print(tree_to_list(search_any_tree(root, 2)))   # [2, 1, 3]
print(tree_to_list(search_any_tree(root, 5)))   # []

[2, 1, 3]
[]


### 4.1. Đếm số node đã thăm — thấy nút thắt

In [9]:
def visits_any_tree(root, val):
    visited = []
    def dfs(node):
        if node is None:
            return None
        visited.append(node.val)
        if node.val == val:
            return node
        return dfs(node.left) or dfs(node.right)
    dfs(root)
    return visited

big_bst = build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13])
print("Tìm 13, duyệt toàn bộ thăm:", visits_any_tree(big_bst, 13))

Tìm 13, duyệt toàn bộ thăm: [8, 3, 1, 6, 4, 7, 10, 14, 13]


**Nút thắt:** tìm 13 mà phải đi qua cả nhánh trái (3, 1, 6, 4, 7) — nơi chắc chắn **không thể** có 13 vì mọi giá trị bên trái 8 đều `< 8`. Ta lãng phí thông tin của BST.

## 5. Insight — mỗi bước bỏ hẳn một nửa cây

Đứng ở node có giá trị `v`:
- `val == v` → tìm thấy.
- `val < v` → nếu có, `val` chỉ có thể nằm ở **cây con trái** → đi trái.
- `val > v` → chỉ có thể ở **cây con phải** → đi phải.

Giống **binary search** trên mảng đã sắp xếp: mỗi lần so sánh loại bỏ một phần. Đường đi chỉ là **một đường từ gốc xuống**, dài tối đa bằng chiều cao `h`.

In [10]:
v, val = 8, 13
print("val < v ?", val < v, "-> đi", "trái" if val < v else "phải")
v = 10
print("val < v ?", val < v, "-> đi", "trái" if val < v else "phải")
v = 14
print("val < v ?", val < v, "-> đi", "trái" if val < v else "phải")
v = 13
print("val == v ?", val == v, "-> tìm thấy")

val < v ? False -> đi phải
val < v ? False -> đi phải
val < v ? True -> đi trái
val == v ? True -> tìm thấy


## 6. Cách 2 — Đệ quy theo tính chất BST

Ghép từng đoạn:
1. Base case: `root is None` → không tìm thấy → `None`. `root.val == val` → trả `root`.
2. `val < root.val` → `return searchBST(root.left, val)`.
3. Ngược lại → `return searchBST(root.right, val)`.

Chú ý: phải **`return`** kết quả của lời gọi đệ quy, nếu không kết quả bị mất.

In [11]:
def search_recursive(root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
    if root is None or root.val == val:
        return root
    if val < root.val:
        return search_recursive(root.left, val)
    return search_recursive(root.right, val)

root = build_tree([4, 2, 7, 1, 3])
print(tree_to_list(search_recursive(root, 2)))
print(tree_to_list(search_recursive(root, 7)))
print(search_recursive(root, 5))

[2, 1, 3]
[7]
None


`root is None or root.val == val`: nhờ **short-circuit** (đánh giá ngắn mạch), nếu `root is None` là `True` thì Python không đánh giá vế sau → không bị lỗi `None.val`. Cả hai trường hợp đều `return root` (là `None` hoặc node tìm thấy).

## 7. Cách 3 — Vòng lặp `while` (lời giải chính, O(1) bộ nhớ)

Đệ quy ở đây là **đệ quy đuôi** (lời gọi đệ quy là việc cuối cùng), nên đổi thành vòng lặp rất tự nhiên: thay vì gọi hàm, ta **di chuyển con trỏ** `node`.

In [12]:
def search_iterative(root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
    node = root
    while node is not None and node.val != val:
        node = node.left if val < node.val else node.right
    return node

root = build_tree([4, 2, 7, 1, 3])
print(tree_to_list(search_iterative(root, 2)))
print(search_iterative(root, 5))

[2, 1, 3]
None


`A if điều_kiện else B` là **biểu thức điều kiện** (ternary): cho `A` nếu điều kiện đúng, ngược lại `B`.

## 8. TRACE từng bước

Cây ví dụ lớn hơn:
```text
            8
          /   \
         3     10
        / \      \
       1   6      14
          / \    /
         4   7  13
```

**Tìm `val = 7`:**

| Bước | node.val | So sánh | Quyết định |
|---|:---:|---|---|
| 1 | 8 | 7 < 8 | đi trái |
| 2 | 3 | 7 > 3 | đi phải |
| 3 | 6 | 7 > 6 | đi phải |
| 4 | 7 | 7 == 7 | **trả node 7** |

**Tìm `val = 5`:** 8 (trái) → 3 (phải) → 6 (trái) → 4 (phải) → `None` → trả `None`.

In [13]:
def trace_search(root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
    node, step = root, 1
    while node is not None:
        if val == node.val:
            print(f"Bước {step}: node={node.val}, {val} == {node.val} -> TÌM THẤY")
            return node
        direction = "trái" if val < node.val else "phải"
        sign = "<" if val < node.val else ">"
        print(f"Bước {step}: node={node.val}, {val} {sign} {node.val} -> đi {direction}")
        node = node.left if val < node.val else node.right
        step += 1
    print(f"Bước {step}: node=None -> KHÔNG có {val}")
    return None

big_bst = build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13])
res = trace_search(big_bst, 7)
print("subtree:", tree_to_list(res))
print()
res = trace_search(big_bst, 5)
print("subtree:", tree_to_list(res))
print()
res = trace_search(big_bst, 13)
print("subtree:", tree_to_list(res))

Bước 1: node=8, 7 < 8 -> đi trái
Bước 2: node=3, 7 > 3 -> đi phải
Bước 3: node=6, 7 > 6 -> đi phải
Bước 4: node=7, 7 == 7 -> TÌM THẤY
subtree: [7]

Bước 1: node=8, 5 < 8 -> đi trái
Bước 2: node=3, 5 > 3 -> đi phải
Bước 3: node=6, 5 < 6 -> đi trái
Bước 4: node=4, 5 > 4 -> đi phải
Bước 5: node=None -> KHÔNG có 5
subtree: []

Bước 1: node=8, 13 > 8 -> đi phải
Bước 2: node=10, 13 > 10 -> đi phải
Bước 3: node=14, 13 < 14 -> đi trái
Bước 4: node=13, 13 == 13 -> TÌM THẤY
subtree: [13]


So với duyệt toàn bộ: tìm 13 chỉ thăm 8 → 10 → 14 → 13 (4 node) thay vì 9 node.

In [14]:
def visits_bst(root, val):
    visited, node = [], root
    while node is not None:
        visited.append(node.val)
        if node.val == val:
            break
        node = node.left if val < node.val else node.right
    return visited

for v in [13, 7, 5, 1]:
    print(f"val={v:2} | toàn bộ: {visits_any_tree(big_bst, v)} | BST: {visits_bst(big_bst, v)}")

val=13 | toàn bộ: [8, 3, 1, 6, 4, 7, 10, 14, 13] | BST: [8, 10, 14, 13]
val= 7 | toàn bộ: [8, 3, 1, 6, 4, 7] | BST: [8, 3, 6, 7]
val= 5 | toàn bộ: [8, 3, 1, 6, 4, 7, 10, 14, 13] | BST: [8, 3, 6, 4]
val= 1 | toàn bộ: [8, 3, 1] | BST: [8, 3, 1]


## 9. Code nộp LeetCode

Bản vòng lặp (khuyên dùng):

In [15]:
class Solution:
    def searchBST(self, root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
        while root and root.val != val:
            root = root.left if val < root.val else root.right
        return root

sol = Solution()
print(tree_to_list(sol.searchBST(build_tree([4, 2, 7, 1, 3]), 2)))   # [2, 1, 3]
print(tree_to_list(sol.searchBST(build_tree([4, 2, 7, 1, 3]), 5)))   # []

[2, 1, 3]
[]


Bản đệ quy (cũng được chấp nhận):

```python
class Solution:
    def searchBST(self, root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
        if root is None or root.val == val:
            return root
        if val < root.val:
            return self.searchBST(root.left, val)
        return self.searchBST(root.right, val)
```

Chú ý trong class phải gọi `self.searchBST(...)`, không phải `searchBST(...)`.

`while root and ...`: một node là "truthy" (đúng) còn `None` là "falsy" (sai), nên `while root` tương đương `while root is not None` ở đây.

## 10. Vì sao đúng?

**Invariant:** *Nếu `val` có trong cây ban đầu, thì nó nằm trong cây con gốc tại `node` hiện tại.*

- Ban đầu `node = root`: đúng hiển nhiên.
- Mỗi bước, nếu `val < node.val`, tính chất BST nói mọi giá trị ở cây con phải và chính `node` đều `≥ node.val > val` → `val` không thể ở đó → nếu có thì ở cây con trái. Tương tự với nhánh phải. Invariant được giữ.
- Vòng lặp dừng khi `node.val == val` (trả đúng node) hoặc `node is None` (cây con rỗng → theo invariant, `val` không có trong cây → trả `None` đúng).

**Kết thúc:** mỗi bước xuống sâu thêm 1 tầng, cây hữu hạn → tối đa `h + 1` bước.

Vì giá trị trong BST theo đề là phân biệt, node tìm thấy là duy nhất.

## 11. Độ phức tạp và so sánh

Gọi `n` = số node, `h` = chiều cao cây (số cạnh trên đường dài nhất từ gốc).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Duyệt toàn bộ (DFS) | `O(n)` | `O(h)` (call stack) | Không dùng BST, đúng với mọi cây |
| 2. Đệ quy BST | `O(h)` | `O(h)` (call stack) | Ngắn, dễ đọc |
| 3. Vòng lặp BST | `O(h)` | `O(1)` | **Nên nộp** |

- Cây **cân bằng**: `h ≈ log₂ n` → `O(log n)`. Với `n = 5000`, `h` khoảng 12.
- Cây **lệch** (ví dụ chèn tăng dần 1, 2, 3, ...): `h = n - 1` → `O(n)` — trường hợp xấu nhất.

In [16]:
import math

def height(root):
    if root is None:
        return -1
    return 1 + max(height(root.left), height(root.right))

def insert(root, v):
    if root is None:
        return TreeNode(v)
    if v < root.val:
        root.left = insert(root.left, v)
    else:
        root.right = insert(root.right, v)
    return root

def sorted_to_bst(arr):
    if not arr:
        return None
    mid = len(arr) // 2
    return TreeNode(arr[mid], sorted_to_bst(arr[:mid]), sorted_to_bst(arr[mid + 1:]))

balanced = sorted_to_bst(list(range(1, 1024)))
skewed = None
for v in range(1, 200):
    skewed = insert(skewed, v)
print("cân bằng, n=1023, h =", height(balanced), "| log2(1024) =", math.log2(1024))
print("lệch,     n=199,  h =", height(skewed))
print("số node thăm khi tìm 199 trên cây lệch:", len(visits_bst(skewed, 199)))

cân bằng, n=1023, h = 9 | log2(1024) = 10.0
lệch,     n=199,  h = 198
số node thăm khi tìm 199 trên cây lệch: 199


## 12. Bộ test — mẫu, biên, phản ví dụ, đối chiếu

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:30} PASS -> {actual}")

sol = Solution()
t = lambda vals, v: tree_to_list(sol.searchBST(build_tree(vals), v))
check("mẫu 1", t([4, 2, 7, 1, 3], 2), [2, 1, 3])
check("mẫu 2 (không có)", t([4, 2, 7, 1, 3], 5), [])
check("tìm gốc", t([4, 2, 7, 1, 3], 4), [4, 2, 7, 1, 3])
check("tìm lá", t([4, 2, 7, 1, 3], 3), [3])
check("cây 1 node, có", t([1], 1), [1])
check("cây 1 node, không", t([1], 2), [])
check("nhỏ hơn mọi node", t([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13], 0), [])
check("lớn hơn mọi node", t([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13], 99), [])
check("node giữa có cây con", t([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13], 6), [6, 4, 7])

mẫu 1                          PASS -> [2, 1, 3]
mẫu 2 (không có)               PASS -> []
tìm gốc                        PASS -> [4, 2, 7, 1, 3]
tìm lá                         PASS -> [3]
cây 1 node, có                 PASS -> [1]
cây 1 node, không              PASS -> []
nhỏ hơn mọi node               PASS -> []
lớn hơn mọi node               PASS -> []
node giữa có cây con           PASS -> [6, 4, 7]


Kiểm tra thêm: kết quả phải là **cùng đối tượng** trong cây (không tạo node mới). Dùng `is` để so sánh danh tính đối tượng.

In [18]:
root = build_tree([4, 2, 7, 1, 3])
print(sol.searchBST(root, 2) is root.left)    # True: chính node 2 trong cây

True


In [19]:
import random
rng = random.Random(700)
for _ in range(300):
    vals = rng.sample(range(1, 60), rng.randint(1, 25))
    r = None
    for v in vals:
        r = insert(r, v)
    for target in range(0, 62):
        a = search_any_tree(r, target)
        assert search_recursive(r, target) is a
        assert search_iterative(r, target) is a
        assert sol.searchBST(r, target) is a
print("300 BST ngẫu nhiên × 62 giá trị: 4 cách trả về cùng node")

300 BST ngẫu nhiên × 62 giá trị: 4 cách trả về cùng node


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên `return` khi gọi đệ quy

In [20]:
def wrong_no_return(root, val):
    if root is None or root.val == val:
        return root
    if val < root.val:
        wrong_no_return(root.left, val)      # thiếu return
    else:
        wrong_no_return(root.right, val)

print("sai:", wrong_no_return(build_tree([4, 2, 7, 1, 3]), 2))
print("đúng:", search_recursive(build_tree([4, 2, 7, 1, 3]), 2))

sai: None
đúng: TreeNode(2)


Vì sao sai: kết quả của lời gọi con bị bỏ đi, hàm chạy hết và ngầm trả `None`. Sửa: `return wrong_no_return(...)`.

### Lỗi 2 — Đi ngược chiều (trái/phải)

In [21]:
def wrong_direction(root, val):
    while root and root.val != val:
        root = root.right if val < root.val else root.left   # đảo ngược
    return root

print("sai:", wrong_direction(build_tree([4, 2, 7, 1, 3]), 2))

sai: None


Vì sao sai: giá trị **nhỏ hơn** nằm bên **trái**. Mẹo nhớ: inorder (trái → gốc → phải) tăng dần, nên "nhỏ = trái".

### Lỗi 3 — Trả về giá trị thay vì node
`return root.val` → LeetCode cần `TreeNode`, trả số sẽ sai kiểu. Đề yêu cầu "return the subtree rooted with that node".

### Lỗi 4 — Truy cập `root.val` khi `root` là `None`
Viết `while root.val != val:` sẽ lỗi `AttributeError: 'NoneType' object has no attribute 'val'` khi không tìm thấy. Luôn kiểm `root is not None` **trước**.

In [22]:
def wrong_none_check(root, val):
    while root.val != val:
        root = root.left if val < root.val else root.right
    return root

try:
    wrong_none_check(build_tree([4, 2, 7, 1, 3]), 5)
except AttributeError as e:
    print("Lỗi:", e)

Lỗi: 'NoneType' object has no attribute 'val'


### Lỗi 5 — Dùng duyệt toàn bộ cây
Vẫn AC (Accepted) vì `n ≤ 5000`, nhưng không đạt mục tiêu học của bài; phỏng vấn sẽ hỏi "đã dùng tính chất BST chưa?".

## 14. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu:** đề nói "binary search tree" + tìm / chèn / xóa / min / max / floor / ceiling / k-th → nghĩ tới **đi một đường từ gốc**, so sánh để chọn trái/phải, `O(h)`.

**Template đi xuống BST:**
```python
node = root
while node:
    if cond_found(node): ...
    node = node.left if key < node.val else node.right
```

Bài liên quan: 701 Insert into a BST, 450 Delete Node in a BST, 235 Lowest Common Ancestor of a BST, 270 Closest BST Value, 98 Validate BST.

### 14.1. Biến thể A — giá trị nhỏ nhất / lớn nhất
Min: đi trái tới cùng. Max: đi phải tới cùng.

In [23]:
def bst_min(root):
    while root.left:
        root = root.left
    return root.val

def bst_max(root):
    while root.right:
        root = root.right
    return root.val

print(bst_min(big_bst), bst_max(big_bst))

1 14


### 14.2. Biến thể B — floor: giá trị lớn nhất `<= x`

Khi đi xuống, mỗi lần gặp `node.val <= x` thì ghi nhận ứng viên và thử đi **phải** (tìm ứng viên lớn hơn); nếu `node.val > x` thì đi **trái**.

In [24]:
def bst_floor(root, x):
    ans = None
    while root:
        if root.val == x:
            return x
        if root.val < x:
            ans = root.val
            root = root.right
        else:
            root = root.left
    return ans

for x in [5, 9, 0, 20, 13]:
    print(f"floor({x}) =", bst_floor(big_bst, x))

floor(5) = 4
floor(9) = 8
floor(0) = None
floor(20) = 14
floor(13) = 13


## 15. Bài tập tự luyện

**Bài 1 (tính tay).** Trên cây `big_bst` (gốc 8), liệt kê các node thăm khi tìm `val = 4`. Gán list vào `ex1_path`.

**Bài 2 (TODO).** `ex2_search_rec(root, val)`: tự viết bản đệ quy.

**Bài 3 (TODO).** `ex3_search_iter(root, val)`: tự viết bản vòng lặp.

**Bài 4 (biến thể).** `ex4_contains(root, val) -> bool`: trả `True/False`.

**Bài 5 (biến thể).** `ex5_ceiling(root, x)`: giá trị nhỏ nhất `>= x` (hoặc `None`).

**Bài 6 (mini challenge).** `ex6_insert(root, val)`: chèn `val` (chưa có trong cây) vào BST và trả về gốc (LeetCode 701), **không dùng đệ quy**.

In [25]:
ex1_path = None   # TODO

def ex2_search_rec(root, val):
    # TODO
    pass

def ex3_search_iter(root, val):
    # TODO
    pass

# assert ex1_path == [...]
# assert tree_to_list(ex2_search_rec(build_tree([4, 2, 7, 1, 3]), 2)) == [2, 1, 3]
# assert ex3_search_iter(build_tree([4, 2, 7, 1, 3]), 5) is None

In [26]:
def ex4_contains(root, val) -> bool:
    # TODO
    pass

def ex5_ceiling(root, x):
    # TODO
    pass

def ex6_insert(root, val):
    # TODO: tìm cha phù hợp rồi gắn node mới
    pass

# assert ex4_contains(big_bst, 13) is True
# assert ex5_ceiling(big_bst, 5) == 6
# assert inorder(ex6_insert(build_tree([4, 2, 7, 1, 3]), 5)) == [1, 2, 3, 4, 5, 7]

## 16. Gợi ý 3 mức

- **Mức 1 (hướng nghĩ):** mọi bài đều "so sánh với node hiện tại rồi đi trái hoặc phải".
- **Mức 2 (cấu trúc):** Bài 5 là ngược của floor: khi `node.val >= x` ghi nhận ứng viên và đi **trái**. Bài 6: đi xuống tới khi con cần đi là `None`, gắn node mới vào đó. Nhớ trường hợp `root is None`.
- **Mức 3 (gần code):** Bài 6:
  ```text
  if root is None: return TreeNode(val)
  node = root
  while True:
      if val < node.val:
          if node.left is None: node.left = TreeNode(val); break
          node = node.left
      else: (tương tự với right)
  return root
  ```

## 17. Đáp án tham khảo

Bài 1: tìm 4 → 8 (4<8 trái) → 3 (4>3 phải) → 6 (4<6 trái) → 4 (bằng). `ex1_path = [8, 3, 6, 4]`.

In [27]:
ans_ex1_path = [8, 3, 6, 4]

def ans_ex2_search_rec(root, val):
    if root is None or root.val == val:
        return root
    return ans_ex2_search_rec(root.left if val < root.val else root.right, val)

def ans_ex3_search_iter(root, val):
    while root and root.val != val:
        root = root.left if val < root.val else root.right
    return root

def ans_ex4_contains(root, val):
    return ans_ex3_search_iter(root, val) is not None

def ans_ex5_ceiling(root, x):
    ans = None
    while root:
        if root.val == x:
            return x
        if root.val > x:
            ans = root.val
            root = root.left
        else:
            root = root.right
    return ans

def ans_ex6_insert(root, val):
    if root is None:
        return TreeNode(val)
    node = root
    while True:
        if val < node.val:
            if node.left is None:
                node.left = TreeNode(val)
                break
            node = node.left
        else:
            if node.right is None:
                node.right = TreeNode(val)
                break
            node = node.right
    return root

assert ans_ex1_path == visits_bst(big_bst, 4)
assert tree_to_list(ans_ex2_search_rec(build_tree([4, 2, 7, 1, 3]), 2)) == [2, 1, 3]
assert ans_ex3_search_iter(build_tree([4, 2, 7, 1, 3]), 5) is None
assert ans_ex4_contains(big_bst, 13) is True and ans_ex4_contains(big_bst, 5) is False
assert ans_ex5_ceiling(big_bst, 5) == 6 and ans_ex5_ceiling(big_bst, 15) is None
assert inorder(ans_ex6_insert(build_tree([4, 2, 7, 1, 3]), 5)) == [1, 2, 3, 4, 5, 7]
for _ in range(200):
    vals = rng.sample(range(1, 80), rng.randint(1, 20))
    r = None
    for v in vals:
        r = ans_ex6_insert(r, v)
    assert inorder(r) == sorted(vals)
    for x in range(0, 82):
        assert ans_ex5_ceiling(r, x) == min((v for v in vals if v >= x), default=None)
print("Đáp án tham khảo: PASS toàn bộ")

Đáp án tham khảo: PASS toàn bộ


## 18. Template nhớ nhanh và checklist

```python
class Solution:
    def searchBST(self, root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
        while root and root.val != val:
            root = root.left if val < root.val else root.right
        return root
```

Câu thần chú: **"Nhỏ đi trái, lớn đi phải, bằng thì trả, hết đường thì None."**

### Checklist trước khi submit
- [ ] Kiểm `root` khác `None` **trước** khi đọc `root.val`.
- [ ] `val < root.val` → trái; `val > root.val` → phải.
- [ ] Trả về **node**, không phải `root.val`.
- [ ] Bản đệ quy: có `return` trước lời gọi đệ quy và dùng `self.searchBST`.
- [ ] Test: tìm gốc, tìm lá, không tồn tại (nhỏ hơn mọi node / lớn hơn mọi node), cây 1 node.
- [ ] Time `O(h)` (cân bằng `O(log n)`, lệch `O(n)`), Space `O(1)` với vòng lặp.